# Alberta Carbon Sequestration Agreement — Exploratory Data Analysis

This notebook provides an initial inspection of the `ABCarbonSequestrationAgreement` spatial dataset.

The immediate objective is to determine:

- what spatial features are contained in the file;
- what attributes are available for each feature;
- what coordinate reference system and geometry types are used;
- the spatial extent and basic structure of the dataset; and
- whether the layer is suitable for integration into the Geospatial-CANOE storage-data workflow.

At this stage, the dataset should be treated as a source dataset requiring inspection rather than as a model-ready representation of geological CO₂ storage capacity or injectivity.

## Dataset Location and Exploration Status

The source dataset is currently stored locally at:

`C:\Users\aviga\Research\potential data\Storage\ABCarbonSequestrationAgreement`

The complete ESRI Shapefile dataset is available, including:

- `.shp` — feature geometry;
- `.shx` — geometry index;
- `.dbf` — attribute table;
- `.prj` — coordinate reference system definition;
- `.cpg` — character encoding information; and
- `.sbn` / `.sbx` — spatial index files.

This notebook is an exploratory inspection of the source dataset prior to formal ingestion into the Geospatial-CANOE data architecture.

The immediate purpose is to understand the dataset's structure, semantics, spatial representation, and potential relevance to CO₂ storage modelling. No transformations into a standardized bronze-layer representation will be performed at this stage.

Following this exploratory assessment, the source data and associated metadata can be incorporated into a more formal bronze-layer workflow with explicit provenance, validation, naming, and storage conventions.

In [24]:
from pathlib import Path

import geopandas as gpd
import pandas as pd

# ---------------------------------------------------------------------------
# Source path
# ---------------------------------------------------------------------------

SOURCE_DIR = Path(r"C:\Users\aviga\Research\potential data\Storage\AB Agreement")
SHAPEFILE_PATH = SOURCE_DIR / "ABCarbonSequestrationAgreement.shp"

# ---------------------------------------------------------------------------
# Basic file check
# ---------------------------------------------------------------------------

print(f"Shapefile exists: {SHAPEFILE_PATH.exists()}")
print(f"Source directory: {SOURCE_DIR}")

# List all associated shapefile components
print("\nAssociated files:")
for path in sorted(SOURCE_DIR.glob("ABCarbonSequestrationAgreement.*")):
    print(f"  - {path.name}")

Shapefile exists: True
Source directory: C:\Users\aviga\Research\potential data\Storage\AB Agreement

Associated files:
  - ABCarbonSequestrationAgreement.cpg
  - ABCarbonSequestrationAgreement.dbf
  - ABCarbonSequestrationAgreement.prj
  - ABCarbonSequestrationAgreement.sbn
  - ABCarbonSequestrationAgreement.sbx
  - ABCarbonSequestrationAgreement.shp
  - ABCarbonSequestrationAgreement.shp.xml
  - ABCarbonSequestrationAgreement.shx


In [2]:
# ---------------------------------------------------------------------------
# Load dataset
# ---------------------------------------------------------------------------

gdf = gpd.read_file(SHAPEFILE_PATH)

print(f"Features: {len(gdf):,}")
print(f"Columns: {len(gdf.columns):,}")
print(f"CRS: {gdf.crs}")
print(f"Geometry types: {gdf.geom_type.unique().tolist()}")

display(gdf.head())

Features: 34
Columns: 19
CRS: EPSG:3400
Geometry types: ['MultiPolygon', 'Polygon', None]


,AGREETYPE,AGREENO,TRACT,MINTYPE,AGGROUP,STATUS,VINTAGE,DESREP,ZONE,ORGAREA,AGREEAREA,TERMDATE,CONTDATE,CUREXPTXT,OBJECTID_1,Longitude,Latitude,Phase,geometry
0,058,5822120014,00,OTHER,AGREEMENT,ACTIVE,None,TIDEWATER MIDSTREAM AND INFRASTRUCTURE LTD.,PORE SPACE BELOW THE TOP OF THE WABAMUN GRP TO...,130944.0,130944.0,2022/12/01,None,2027/12/01,1442392,4660489,2063431,2,"MULTIPOLYGON (((500023.269 5755372.314, 499945..."
1,058,5822120004,00,OTHER,AGREEMENT,ACTIVE,None,TIDEWATER MIDSTREAM AND INFRASTRUCTURE LTD.,PORE SPACE IN THE WABAMUN GRP,497152.0,497152.0,2022/12/01,None,2027/12/01,1443171,4688243,2184405,2,"MULTIPOLYGON (((504286.26 5924072.267, 504284...."
2,058,5822120012,00,OTHER,AGREEMENT,ACTIVE,None,WEST LAKE ENERGY CORP.,PORE SPACE IN THE RUNDLE GRP,63360.0,63360.0,2022/12/01,None,2027/12/01,1441572,4639703,1732093,2,"MULTIPOLYGON (((583527.778 5448566.333, 583122..."
3,058,5822120010,00,OTHER,AGREEMENT,ACTIVE,None,KIWETINOHK ENERGY CORP.,PORE SPACE BELOW THE TOP OF THE ELK POINT GRP ...,82944.0,82944.0,2022/12/01,None,2027/12/01,1440808,4655002,2333189,2,"MULTIPOLYGON (((384946.796 6029075.115, 384936..."
4,058,5822120007,00,OTHER,AGREEMENT,ACTIVE,None,KIWETINOHK ENERGY CORP.,PORE SPACE BELOW THE TOP OF THE ELK POINT GRP ...,146176.0,146176.0,2022/12/01,None,2027/12/01,1440807,4739697,2321452,2,"MULTIPOLYGON (((472655.651 6066516.09, 472653...."


In [4]:
# ---------------------------------------------------------------------------
# Inspect attribute schema
# ---------------------------------------------------------------------------

schema = pd.DataFrame({
    "column": gdf.columns,
    "dtype": gdf.dtypes.astype(str),
    "non_null": gdf.notna().sum().values,
    "null": gdf.isna().sum().values,
    "unique_values": [
        gdf[col].nunique(dropna=True)
        if col != "geometry"
        else gdf.geom_type.nunique(dropna=True)
        for col in gdf.columns
    ],
})

display(schema)

print("Column names:")
for i, column in enumerate(gdf.columns):
    print(f"{i:>2}: {column}")

,column,dtype,non_null,null,unique_values
AGREETYPE,AGREETYPE,object,33,1,2
AGREENO,AGREENO,object,33,1,30
TRACT,TRACT,object,33,1,3
MINTYPE,MINTYPE,object,33,1,1
AGGROUP,AGGROUP,object,33,1,1
STATUS,STATUS,object,33,1,1
VINTAGE,VINTAGE,object,0,34,0
DESREP,DESREP,object,33,1,20
ZONE,ZONE,object,33,1,15
ORGAREA,ORGAREA,float64,34,0,29


Column names:
 0: AGREETYPE
 1: AGREENO
 2: TRACT
 3: MINTYPE
 4: AGGROUP
 5: STATUS
 6: VINTAGE
 7: DESREP
 8: ZONE
 9: ORGAREA
10: AGREEAREA
11: TERMDATE
12: CONTDATE
13: CUREXPTXT
14: OBJECTID_1
15: Longitude
16: Latitude
17: Phase
18: geometry


In [5]:
# ---------------------------------------------------------------------------
# Inspect categorical attribute values
# ---------------------------------------------------------------------------

categorical_columns = [
    "AGREETYPE",
    "TRACT",
    "MINTYPE",
    "AGGROUP",
    "STATUS",
    "DESREP",
    "ZONE",
    "TERMDATE",
    "CUREXPTXT",
    "Phase",
]

for column in categorical_columns:
    print("\n" + "=" * 70)
    print(column)
    print("=" * 70)

    print(
        gdf[column]
        .value_counts(dropna=False)
        .to_string()
    )


AGREETYPE
AGREETYPE
058     27
059      6
None     1

TRACT
TRACT
00      27
01       3
02       3
None     1

MINTYPE
MINTYPE
OTHER    33
None      1

AGGROUP
AGGROUP
AGREEMENT    33
None          1

STATUS
STATUS
ACTIVE    33
None       1

DESREP
DESREP
SHELL CANADA LIMITED                           7
KIWETINOHK ENERGY CORP.                        2
BISON LOW CARBON VENTURES INC.                 2
ENBRIDGE WABAMUN HUB LTD.                      2
TIDEWATER MIDSTREAM AND INFRASTRUCTURE LTD.    2
BOW VALLEY CARBON LTD.                         2
VAULT 44.01 LTD.                               2
ENHANCE ENERGY INC.                            2
WEST LAKE ENERGY CORP.                         1
WOLF CENTRAL ALBERTA CARBON HUB INC.           1
RECONCILIATION ENERGY TRANSITION INC.          1
NORTHRIVER MIDSTREAM GP NET ZERO INC.          1
WOLF CARBON HUB GP INC.                        1
MEDICINE HAT, CITY OF                          1
CANADIAN NATURAL RESOURCES LIMITED             1
PEMBINA 

In [6]:
# ---------------------------------------------------------------------------
# Inspect null record and repeated agreement numbers
# ---------------------------------------------------------------------------

print("Rows with missing agreement number or geometry:")
display(
    gdf[
        gdf["AGREENO"].isna() |
        gdf.geometry.isna()
    ]
)

print("\nRepeated agreement numbers:")
agreement_counts = (
    gdf["AGREENO"]
    .value_counts(dropna=True)
)

repeated_agreements = agreement_counts[agreement_counts > 1].index

display(
    gdf[
        gdf["AGREENO"].isin(repeated_agreements)
    ][
        [
            "AGREENO",
            "TRACT",
            "AGREETYPE",
            "STATUS",
            "DESREP",
            "ZONE",
            "ORGAREA",
            "AGREEAREA",
            "TERMDATE",
            "CUREXPTXT",
            "Phase",
            "geometry",
        ]
    ]
    .sort_values(["AGREENO", "TRACT"])
)

Rows with missing agreement number or geometry:


,AGREETYPE,AGREENO,TRACT,MINTYPE,AGGROUP,STATUS,VINTAGE,DESREP,ZONE,ORGAREA,AGREEAREA,TERMDATE,CONTDATE,CUREXPTXT,OBJECTID_1,Longitude,Latitude,Phase,geometry
33,None,None,None,None,None,None,None,None,None,0.0,0.0,None,None,None,0,0,0,0,None



Repeated agreement numbers:


,AGREENO,TRACT,AGREETYPE,STATUS,DESREP,ZONE,ORGAREA,AGREEAREA,TERMDATE,CUREXPTXT,Phase,geometry
7,5822100007,01,058,ACTIVE,ENBRIDGE WABAMUN HUB LTD.,PORE SPACE IN THE WINTERBURN GRP PORE SPACE IN...,211072.0,209711.482,2022/10/01,2027/10/01,1,"MULTIPOLYGON (((548562.704 5910988.924, 548409..."
8,5822100007,02,058,ACTIVE,ENBRIDGE WABAMUN HUB LTD.,PORE SPACE IN THE BASAL SANDSTONE UNIT,211072.0,209711.482,2022/10/01,2027/10/01,1,"MULTIPOLYGON (((572356.952 5946861.046, 571954..."
30,5822100008,01,058,ACTIVE,ENHANCE ENERGY INC.,PORE SPACE IN THE WOODBEND GRP,544512.0,536441.426,2022/10/01,2027/10/01,1,"MULTIPOLYGON (((615479.682 5834314.731, 615364..."
31,5822100008,02,058,ACTIVE,ENHANCE ENERGY INC.,PORE SPACE IN THE WOODBEND GRP EXCEPTING PORE ...,544512.0,536441.426,2022/10/01,2027/10/01,1,"POLYGON ((599082.378 5793421.656, 598683.149 5..."
15,5822120003,01,058,ACTIVE,BOW VALLEY CARBON LTD.,PORE SPACE BELOW THE TOP OF THE WINTERBURN GRP...,106048.0,106048.000,2022/12/01,2027/12/01,2,"MULTIPOLYGON (((555010.451 5677967.989, 554906..."
16,5822120003,02,058,ACTIVE,BOW VALLEY CARBON LTD.,PORE SPACE IN THE WINTERBURN GRP,106048.0,106048.000,2022/12/01,2027/12/01,2,"MULTIPOLYGON (((532382.913 5687509.625, 531980..."


In [7]:
# ---------------------------------------------------------------------------
# Confirm dataset grain: agreement vs tract
# ---------------------------------------------------------------------------

n_rows = len(gdf)
n_agreements = gdf["AGREENO"].nunique(dropna=True)

agreement_tract_pairs = (
    gdf[["AGREENO", "TRACT"]]
    .dropna()
    .drop_duplicates()
)

n_agreement_tracts = len(agreement_tract_pairs)

print(f"Total rows:                 {n_rows:,}")
print(f"Unique agreement numbers:   {n_agreements:,}")
print(f"Unique agreement-tract pairs: {n_agreement_tracts:,}")

print("\nAgreement tract counts:")

display(
    gdf.dropna(subset=["AGREENO"])
       .groupby("AGREENO")
       .agg(
           n_rows=("TRACT", "size"),
           n_tracts=("TRACT", "nunique"),
           representative=("DESREP", "first"),
           n_zones=("ZONE", "nunique"),
       )
       .sort_values(["n_tracts", "n_zones"], ascending=False)
)

Total rows:                 34
Unique agreement numbers:   30
Unique agreement-tract pairs: 33

Agreement tract counts:


,n_rows,n_tracts,representative,n_zones
AGREENO,,,,
5822100007,2,2,ENBRIDGE WABAMUN HUB LTD.,2
5822100008,2,2,ENHANCE ENERGY INC.,2
5822120003,2,2,BOW VALLEY CARBON LTD.,2
5822100009,1,1,BISON LOW CARBON VENTURES INC.,1
5822100010,1,1,PEMBINA PIPELINE CORPORATION,1
5822100011,1,1,SHELL CANADA LIMITED,1
5822100012,1,1,WOLF CARBON HUB GP INC.,1
5822120001,1,1,VAULT 44.01 LTD.,1
5822120002,1,1,ALBERTA POWER (2000) LTD,1


In [8]:
# ---------------------------------------------------------------------------
# Inspect geometry completeness and validity
# ---------------------------------------------------------------------------

geometry_audit = pd.DataFrame({
    "geometry_type": gdf.geom_type,
    "is_null": gdf.geometry.isna(),
    "is_empty": gdf.geometry.is_empty,
    "is_valid": gdf.geometry.is_valid,
})

print("Geometry summary:")
print(f"  Total records:      {len(gdf):,}")
print(f"  Null geometries:    {gdf.geometry.isna().sum():,}")
print(f"  Empty geometries:   {gdf.geometry.is_empty.sum():,}")
print(f"  Invalid geometries: {(~gdf.geometry.is_valid & gdf.geometry.notna()).sum():,}")

print("\nGeometry types:")
print(gdf.geom_type.value_counts(dropna=False).to_string())

print("\nRecords with missing or invalid geometry:")
display(
    gdf.loc[
        gdf.geometry.isna()
        | gdf.geometry.is_empty
        | (~gdf.geometry.is_valid & gdf.geometry.notna()),
        [
            "AGREENO",
            "TRACT",
            "DESREP",
            "ZONE",
            "OBJECTID_1",
            "Phase",
            "geometry",
        ],
    ]
)

Geometry summary:
  Total records:      34
  Null geometries:    1
  Empty geometries:   0
  Invalid geometries: 0

Geometry types:
MultiPolygon    32
Polygon          1
None             1

Records with missing or invalid geometry:


,AGREENO,TRACT,DESREP,ZONE,OBJECTID_1,Phase,geometry
33,None,None,None,None,0,0,None


In [9]:
# ---------------------------------------------------------------------------
# Inspect CRS units and compare source vs computed polygon area
# ---------------------------------------------------------------------------

print("CRS:")
print(gdf.crs)

print("\nCRS details:")
print(gdf.crs.to_string())
print("Axis information:")

for axis in gdf.crs.axis_info:
    print(
        f"  - {axis.name}: "
        f"unit={axis.unit_name}, "
        f"conversion_factor={axis.unit_conversion_factor}"
    )

# Compute area only for records with geometry
spatial = gdf[gdf.geometry.notna()].copy()

spatial["computed_area_native"] = spatial.geometry.area
spatial["computed_area_km2"] = spatial["computed_area_native"] / 1_000_000

area_comparison = spatial[
    [
        "AGREENO",
        "TRACT",
        "DESREP",
        "ORGAREA",
        "AGREEAREA",
        "computed_area_native",
        "computed_area_km2",
    ]
].copy()

display(area_comparison.head(10))

CRS:
EPSG:3400

CRS details:
EPSG:3400
Axis information:
  - Easting: unit=metre, conversion_factor=1.0
  - Northing: unit=metre, conversion_factor=1.0


,AGREENO,TRACT,DESREP,ORGAREA,AGREEAREA,computed_area_native,computed_area_km2
0,5822120014,00,TIDEWATER MIDSTREAM AND INFRASTRUCTURE LTD.,130944.0,130944.000,1.318617e+09,1318.617360
1,5822120004,00,TIDEWATER MIDSTREAM AND INFRASTRUCTURE LTD.,497152.0,497152.000,5.011443e+09,5011.443407
2,5822120012,00,WEST LAKE ENERGY CORP.,63360.0,63360.000,6.379748e+08,637.974801
3,5822120010,00,KIWETINOHK ENERGY CORP.,82944.0,82944.000,8.312482e+08,831.248238
4,5822120007,00,KIWETINOHK ENERGY CORP.,146176.0,146176.000,1.463384e+09,1463.384114
5,5822120008,00,BISON LOW CARBON VENTURES INC.,66816.0,66816.000,6.746946e+08,674.694643
6,5822100009,00,BISON LOW CARBON VENTURES INC.,70400.0,70400.000,7.102224e+08,710.222401
7,5822100007,01,ENBRIDGE WABAMUN HUB LTD.,211072.0,209711.482,1.421949e+09,1421.948614
8,5822100007,02,ENBRIDGE WABAMUN HUB LTD.,211072.0,209711.482,6.951972e+08,695.197181
9,5822100012,00,WOLF CARBON HUB GP INC.,184064.0,184064.000,1.854107e+09,1854.106788


In [12]:
# ---------------------------------------------------------------------------
# Compare agreement-level source area with summed tract geometry
# ---------------------------------------------------------------------------

agreement_area = (
    spatial
    .groupby("AGREENO")
    .agg(
        n_tracts=("TRACT", "nunique"),
        representative=("DESREP", "first"),
        source_orgarea=("ORGAREA", "first"),
        source_agreearea=("AGREEAREA", "first"),
        summed_geometry_area_ha=("computed_area_ha", "sum"),
    )
    .reset_index()
)

agreement_area["orgarea_pct_diff"] = (
    (agreement_area["source_orgarea"] - agreement_area["summed_geometry_area_ha"])
    / agreement_area["summed_geometry_area_ha"]
    * 100
)

agreement_area["agreearea_pct_diff"] = (
    (agreement_area["source_agreearea"] - agreement_area["summed_geometry_area_ha"])
    / agreement_area["summed_geometry_area_ha"]
    * 100
)

display(
    agreement_area
    .sort_values("n_tracts", ascending=False)
)

,AGREENO,n_tracts,representative,source_orgarea,source_agreearea,summed_geometry_area_ha,orgarea_pct_diff,agreearea_pct_diff
0,5822100007,2,ENBRIDGE WABAMUN HUB LTD.,211072.0,209711.482,2.117146e+05,-0.303512,-0.946131
1,5822100008,2,ENHANCE ENERGY INC.,544512.0,536441.426,5.420883e+05,0.447098,-1.041695
8,5822120003,2,BOW VALLEY CARBON LTD.,106048.0,106048.000,1.072879e+05,-1.155634,-1.155634
2,5822100009,1,BISON LOW CARBON VENTURES INC.,70400.0,70400.000,7.102224e+04,-0.876120,-0.876120
4,5822100011,1,SHELL CANADA LIMITED,1031232.0,1024615.700,1.034155e+06,-0.282691,-0.922469
3,5822100010,1,PEMBINA PIPELINE CORPORATION,950720.0,946246.890,9.545537e+05,-0.401623,-0.870231
5,5822100012,1,WOLF CARBON HUB GP INC.,184064.0,184064.000,1.854107e+05,-0.726322,-0.726322
6,5822120001,1,VAULT 44.01 LTD.,73280.0,73280.000,7.375054e+04,-0.638021,-0.638021
7,5822120002,1,ALBERTA POWER (2000) LTD,1729792.0,1729792.000,1.750867e+06,-1.203667,-1.203667
9,5822120004,1,TIDEWATER MIDSTREAM AND INFRASTRUCTURE LTD.,497152.0,497152.000,5.011443e+05,-0.796645,-0.796645


## Preliminary Dataset Structure

The exploratory checks indicate that the source dataset is organized primarily at the **agreement-tract level** rather than strictly at the agreement level.

Each populated record contains an `AGREENO` agreement identifier and a `TRACT` identifier. Most agreements contain a single tract, while several agreements are represented by multiple tract records. These multi-tract records can also contain different `ZONE` descriptions, indicating that the tract distinction preserves meaningful spatial and/or stratigraphic information.

The observed structure can therefore be represented conceptually as:

**Agreement → one or more tracts → tract geometry and stratigraphic pore-space description**

The source fields `ORGAREA` and `AGREEAREA` appear to be reported in hectares. Direct comparison against geometry-derived areas shows small differences, generally on the order of approximately 1%. For multi-tract agreements, agreement-level comparison requires summing the geometry of all associated tracts before comparison with the source area fields.

This suggests that the source area attributes should be retained as authoritative source metadata, while geometry-derived area should be treated as a separately calculated spatial attribute.

No dissolution or aggregation of tract geometries will be performed during source exploration.

In [13]:
# ---------------------------------------------------------------------------
# Load and inspect shapefile XML metadata
# ---------------------------------------------------------------------------

from xml.etree import ElementTree as ET

XML_PATH = SOURCE_DIR / "ABCarbonSequestrationAgreement.shp.xml"

print(f"XML metadata exists: {XML_PATH.exists()}")
print(f"XML path: {XML_PATH}")

tree = ET.parse(XML_PATH)
root = tree.getroot()

print(f"\nRoot tag: {root.tag}")
print(f"Top-level child elements: {len(root)}")

print("\nTop-level tags:")
for child in root:
    print(f"  - {child.tag}")

XML metadata exists: True
XML path: C:\Users\aviga\Research\potential data\Storage\ABCarbonSequestrationAgreement.shp.xml

Root tag: metadata
Top-level child elements: 14

Top-level tags:
  - Esri
  - mdLang
  - mdChar
  - mdHrLv
  - mdContact
  - distInfo
  - dataIdInfo
  - dqInfo
  - eainfo
  - mdHrLvName
  - refSysInfo
  - spatRepInfo
  - spdoinfo
  - mdDateSt


In [14]:
# ---------------------------------------------------------------------------
# Inspect populated XML metadata by top-level section
# ---------------------------------------------------------------------------

def iter_text_elements(element):
    """Yield populated descendant elements as (tag, text) pairs."""
    for descendant in element.iter():
        text = descendant.text.strip() if descendant.text else ""
        if text:
            yield descendant.tag, text


for section in root:
    print("\n" + "=" * 80)
    print(section.tag)
    print("=" * 80)

    entries = list(iter_text_elements(section))

    for tag, text in entries[:40]:
        print(f"{tag}: {text}")

    if len(entries) > 40:
        print(f"... {len(entries) - 40} additional populated elements")


Esri
CreaDate: 20240419
CreaTime: 11450300
ArcGISFormat: 1.0
SyncOnce: FALSE
Process: CopyFeatures lyr_shp_AmiAgr_All \\GOA\appsdata\GIS_Energy\spatial\spatial_data\DOE\MineralAgreementsProvWide\AmiAgr_All20230206.shp # 0 0 0
Process: UpdateSchema "CIMDATA=<CIMStandardDataConnection xsi:type='typens:CIMStandardDataConnection' xmlns:xsi='http://www.w3.org/2001/XMLSchema-instance' xmlns:xs='http://www.w3.org/2001/XMLSchema' xmlns:typens='http://www.esri.com/schemas/ArcGIS/3.1.0'><WorkspaceConnectionString>DATABASE=Z:\NRCan Projects\NCAF\2.0 Network Development and Optimization\ARCGIS\Layer Data\AB Storage Shapefiles</WorkspaceConnectionString><WorkspaceFactory>Shapefile</WorkspaceFactory><Dataset>CarbonSequestrationAgreement.shp</Dataset><DatasetType>esriDTFeatureClass</DatasetType></CIMStandardDataConnection>" <operationSequence><workflow><AddField><field_name>Longitude</field_name><field_type>LONG</field_type><field_is_nullable>False</field_is_nullable><field_is_required>False</field_

In [15]:
# ---------------------------------------------------------------------------
# Extract attribute definitions from XML metadata
# ---------------------------------------------------------------------------

attributes = []

for attr in root.findall(".//attr"):
    record = {}

    for child in attr:
        text = child.text.strip() if child.text else None

        if child.tag == "attrlabl":
            record["field"] = text
        elif child.tag == "attalias":
            record["alias"] = text
        elif child.tag == "attrtype":
            record["type"] = text
        elif child.tag == "attwidth":
            record["width"] = text
        elif child.tag == "attrdef":
            record["definition"] = text
        elif child.tag == "attrdefs":
            record["definition_source"] = text
        elif child.tag == "udom":
            record["domain"] = text

    if record:
        attributes.append(record)

attribute_metadata = pd.DataFrame(attributes)

display(attribute_metadata)

,field,alias,type,width,definition,definition_source
0,FID,FID,OID,4,Internal feature number.,Esri
1,Shape,Shape,Geometry,0,Feature geometry.,ESRI
2,AGREETYPE,Agreement Type,String,3,NaN,NaN
3,AGREENO,Agreement Number,String,10,NaN,NaN
4,TRACT,TRACT,String,2,Agreement Tract,"Alberta Energy and Minerals, Governement of Al..."
5,MINTYPE,Mineral Type,String,10,NaN,NaN
6,AGGROUP,Agreement Group,String,10,NaN,NaN
7,STATUS,STATUS,String,254,Current status of the agreement - posted (for ...,"Alberta Energy and Minerals, Governement of Al..."
8,VINTAGE,VINTAGE,String,50,NaN,NaN
9,DESREP,Designated Representative,String,150,NaN,NaN


## Source Attribute Definitions

The XML metadata provides authoritative definitions for several key source attributes.

| Field | Source definition |
|---|---|
| `TRACT` | Agreement Tract |
| `STATUS` | Current status of the agreement |
| `DESREP` | Designated Representative |
| `TERMDATE` | Commencement date of the agreement |
| `ORGAREA` | Original deemed agreement area in hectares |
| `AGREEAREA` | Current deemed agreement area in hectares |
| `ZONE` | Mineral substance and zones contained in or excepted from the agreement |

The metadata therefore confirms that the source area fields are reported in **hectares**.

`ORGAREA` represents the original deemed agreement area, while `AGREEAREA` represents the current deemed agreement area. These should be preserved as source-defined administrative attributes rather than replaced by geometry-derived area.

Geometry-derived area can still be calculated independently for spatial validation and downstream analysis.

In [16]:
# ---------------------------------------------------------------------------
# Extract coded-value domains from XML metadata
# ---------------------------------------------------------------------------

domain_rows = []

for attr in root.findall(".//attr"):
    field = attr.findtext("attrlabl")

    # Enumerated domains are usually stored under attrdomv/edom
    for edom in attr.findall(".//edom"):
        value = edom.findtext("edomv")
        definition = edom.findtext("edomvd")
        source = edom.findtext("edomvds")

        domain_rows.append(
            {
                "field": field,
                "value": value,
                "definition": definition,
                "definition_source": source,
            }
        )

domain_metadata = pd.DataFrame(domain_rows)

display(domain_metadata)

""


In [17]:
# ---------------------------------------------------------------------------
# Compare shapefile schema against XML metadata definitions
# ---------------------------------------------------------------------------

xml_fields = set(attribute_metadata["field"].dropna())
shapefile_fields = set(gdf.columns) - {"geometry"}

print("Fields present in shapefile but not defined in XML:")
for field in sorted(shapefile_fields - xml_fields):
    print(f"  - {field}")

print("\nFields defined in XML but not present in shapefile:")
for field in sorted(xml_fields - shapefile_fields):
    print(f"  - {field}")

print("\nFields present in both:")
for field in sorted(shapefile_fields & xml_fields):
    print(f"  - {field}")

Fields present in shapefile but not defined in XML:
  - Latitude
  - Longitude
  - Phase

Fields defined in XML but not present in shapefile:
  - CRTNDATE
  - FID
  - Shape

Fields present in both:
  - AGGROUP
  - AGREEAREA
  - AGREENO
  - AGREETYPE
  - CONTDATE
  - CUREXPTXT
  - DESREP
  - MINTYPE
  - OBJECTID_1
  - ORGAREA
  - STATUS
  - TERMDATE
  - TRACT
  - VINTAGE
  - ZONE


In [19]:
# ---------------------------------------------------------------------------
# Extract lineage/process steps without assuming the parent tag name
# ---------------------------------------------------------------------------

process_rows = []

for element in root.iter():
    step_desc = element.find("stepDesc")
    step_date = element.find("stepDateTm")

    if step_desc is not None:
        process_rows.append(
            {
                "parent_tag": element.tag,
                "description": step_desc.text.strip() if step_desc.text else None,
                "date_time": (
                    step_date.text.strip()
                    if step_date is not None and step_date.text
                    else None
                ),
            }
        )

process_history = pd.DataFrame(process_rows)

display(process_history)

,parent_tag,description,date_time
0,prcStep,Metadata imported and edited.,2024-04-19T00:00:00
1,prcStep,"Attributes are captured directly from AMI, the...",2024-04-19T00:00:00


In [20]:
# ---------------------------------------------------------------------------
# Extract Esri geoprocessing history
# ---------------------------------------------------------------------------

esri_processes = []

for element in root.findall(".//Process"):
    text = element.text.strip() if element.text else None

    if text:
        esri_processes.append(text)

print(f"Esri processing records: {len(esri_processes)}\n")

for i, process in enumerate(esri_processes, start=1):
    print("=" * 80)
    print(f"Process {i}")
    print("=" * 80)
    print(process)
    print()

Esri processing records: 5

Process 1
CopyFeatures lyr_shp_AmiAgr_All \\GOA\appsdata\GIS_Energy\spatial\spatial_data\DOE\MineralAgreementsProvWide\AmiAgr_All20230206.shp # 0 0 0

Process 2
UpdateSchema "CIMDATA=<CIMStandardDataConnection xsi:type='typens:CIMStandardDataConnection' xmlns:xsi='http://www.w3.org/2001/XMLSchema-instance' xmlns:xs='http://www.w3.org/2001/XMLSchema' xmlns:typens='http://www.esri.com/schemas/ArcGIS/3.1.0'><WorkspaceConnectionString>DATABASE=Z:\NRCan Projects\NCAF\2.0 Network Development and Optimization\ARCGIS\Layer Data\AB Storage Shapefiles</WorkspaceConnectionString><WorkspaceFactory>Shapefile</WorkspaceFactory><Dataset>CarbonSequestrationAgreement.shp</Dataset><DatasetType>esriDTFeatureClass</DatasetType></CIMStandardDataConnection>" <operationSequence><workflow><AddField><field_name>Longitude</field_name><field_type>LONG</field_type><field_is_nullable>False</field_is_nullable><field_is_required>False</field_is_required></AddField></workflow></operationSe

In [21]:
# ---------------------------------------------------------------------------
# Validate derived Longitude / Latitude fields
# ---------------------------------------------------------------------------

spatial = gdf[gdf.geometry.notna()].copy()

# Reproject source geometries from EPSG:3400 to Statistics Canada Lambert
spatial_3347 = spatial.to_crs("EPSG:3347")

# Calculate centroids in the same CRS used by the recorded ArcGIS process
spatial["centroid_x_epsg3347"] = spatial_3347.geometry.centroid.x.values
spatial["centroid_y_epsg3347"] = spatial_3347.geometry.centroid.y.values

# Compare against the fields labelled Longitude and Latitude
spatial["longitude_diff_m"] = (
    spatial["Longitude"] - spatial["centroid_x_epsg3347"]
)

spatial["latitude_diff_m"] = (
    spatial["Latitude"] - spatial["centroid_y_epsg3347"]
)

centroid_validation = spatial[
    [
        "AGREENO",
        "TRACT",
        "Longitude",
        "Latitude",
        "centroid_x_epsg3347",
        "centroid_y_epsg3347",
        "longitude_diff_m",
        "latitude_diff_m",
    ]
].copy()

display(centroid_validation.head(10))

print("\nDifference summary (metres):")
display(
    centroid_validation[
        ["longitude_diff_m", "latitude_diff_m"]
    ].describe()
)

,AGREENO,TRACT,Longitude,Latitude,centroid_x_epsg3347,centroid_y_epsg3347,longitude_diff_m,latitude_diff_m
0,5822120014,00,4660489,2063431,4.660489e+06,2.063431e+06,-0.317121,-0.255369
1,5822120004,00,4688243,2184405,4.688243e+06,2.184405e+06,-0.029011,0.200955
2,5822120012,00,4639703,1732093,4.639703e+06,1.732093e+06,0.263416,-0.113260
3,5822120010,00,4655002,2333189,4.655002e+06,2.333189e+06,0.488810,0.333477
4,5822120007,00,4739697,2321452,4.739697e+06,2.321452e+06,-0.316106,-0.014418
5,5822120008,00,4818205,1974355,4.818205e+06,1.974355e+06,0.081232,0.336241
6,5822100009,00,4825154,2220917,4.825154e+06,2.220917e+06,0.391593,-0.149739
7,5822100007,01,4758377,2173388,4.758377e+06,2.173388e+06,-0.199545,0.367947
8,5822100007,02,4805646,2193463,4.805646e+06,2.193463e+06,0.129654,0.171986
9,5822100012,00,4894513,2175938,4.894513e+06,2.175938e+06,0.260009,-0.455953



Difference summary (metres):


,longitude_diff_m,latitude_diff_m
count,33.000000,33.000000
mean,0.013686,-0.021082
std,0.283311,0.327251
min,-0.496637,-0.479828
25%,-0.259534,-0.380435
50%,0.081232,-0.014418
75%,0.229958,0.257533
max,0.488810,0.491819


## Derived Centroid Coordinate Fields

The source shapefile contains fields named `Longitude` and `Latitude`. XML geoprocessing history shows that these fields were added after the original agreement dataset and populated using ArcGIS `CalculateGeometryAttributes` with:

- `Longitude` = `CENTROID_X`
- `Latitude` = `CENTROID_Y`

The calculation was performed in **NAD83(CSRS) / Statistics Canada Lambert**, corresponding to EPSG:3347.

A numerical validation was performed by reprojecting the agreement geometries to EPSG:3347 and recalculating polygon centroids. The stored values differ from the recomputed centroids by less than approximately one metre.

Therefore:

- `Longitude` is not geographic longitude; it is projected centroid easting in metres.
- `Latitude` is not geographic latitude; it is projected centroid northing in metres.

These field names should be treated as misleading source labels. They should be preserved unchanged in a raw/bronze representation for provenance, but should not be propagated under these names into standardized downstream datasets.

In [22]:
# ---------------------------------------------------------------------------
# Explore the derived Phase field
# ---------------------------------------------------------------------------

phase_summary = (
    gdf[
        [
            "AGREENO",
            "TRACT",
            "AGREETYPE",
            "DESREP",
            "ZONE",
            "TERMDATE",
            "CUREXPTXT",
            "Phase",
        ]
    ]
    .sort_values(["Phase", "AGREENO", "TRACT"])
)

print("Phase counts:")
print(gdf["Phase"].value_counts(dropna=False).sort_index().to_string())

print("\nPhase by agreement type:")
display(
    pd.crosstab(
        gdf["Phase"],
        gdf["AGREETYPE"],
        dropna=False,
    )
)

print("\nRecords by phase:")
display(phase_summary)

Phase counts:
Phase
0     1
1     8
2    25

Phase by agreement type:


AGREETYPE,058,059,NaN
Phase,,,
0,0,0,1
1,8,0,0
2,19,6,0



Records by phase:


,AGREENO,TRACT,AGREETYPE,DESREP,ZONE,TERMDATE,CUREXPTXT,Phase
33,None,None,None,None,None,None,None,0
7,5822100007,01,058,ENBRIDGE WABAMUN HUB LTD.,PORE SPACE IN THE WINTERBURN GRP PORE SPACE IN...,2022/10/01,2027/10/01,1
8,5822100007,02,058,ENBRIDGE WABAMUN HUB LTD.,PORE SPACE IN THE BASAL SANDSTONE UNIT,2022/10/01,2027/10/01,1
30,5822100008,01,058,ENHANCE ENERGY INC.,PORE SPACE IN THE WOODBEND GRP,2022/10/01,2027/10/01,1
31,5822100008,02,058,ENHANCE ENERGY INC.,PORE SPACE IN THE WOODBEND GRP EXCEPTING PORE ...,2022/10/01,2027/10/01,1
6,5822100009,00,058,BISON LOW CARBON VENTURES INC.,PORE SPACE IN THE WOODBEND GRP,2022/10/01,2027/10/01,1
27,5822100010,00,058,PEMBINA PIPELINE CORPORATION,PORE SPACE IN THE BASAL SANDSTONE UNIT,2022/10/01,2027/10/01,1
24,5822100011,00,058,SHELL CANADA LIMITED,PORE SPACE IN THE BASAL SANDSTONE UNIT,2022/10/01,2027/10/01,1
9,5822100012,00,058,WOLF CARBON HUB GP INC.,PORE SPACE IN THE BASAL SANDSTONE UNIT,2022/10/01,2027/10/01,1
12,5822120001,00,058,VAULT 44.01 LTD.,PORE SPACE BELOW THE TOP OF THE BASAL SANDSTON...,2022/12/01,2027/12/01,2


In [23]:
# ---------------------------------------------------------------------------
# Test whether Phase corresponds to agreement-date cohorts
# ---------------------------------------------------------------------------

phase_cohorts = (
    gdf[gdf["AGREENO"].notna()]
    .groupby(
        [
            "Phase",
            "AGREETYPE",
            "TERMDATE",
            "CUREXPTXT",
        ],
        dropna=False,
    )
    .agg(
        n_rows=("AGREENO", "size"),
        n_agreements=("AGREENO", "nunique"),
        agreement_numbers=(
            "AGREENO",
            lambda x: ", ".join(sorted(x.astype(str).unique()))
        ),
    )
    .reset_index()
)

display(phase_cohorts)

,Phase,AGREETYPE,TERMDATE,CUREXPTXT,n_rows,n_agreements,agreement_numbers
0,1,058,2022/10/01,2027/10/01,8,6,"5822100007, 5822100008, 5822100009, 5822100010..."
1,2,058,2022/12/01,2027/12/01,18,17,"5822120001, 5822120002, 5822120003, 5822120004..."
2,2,058,2023/01/01,2028/01/01,1,1,5823010001
3,2,059,2011/05/27,2026/05/27,6,6,"5911050001, 5911050002, 5911050003, 5911050004..."


In [25]:
# ---------------------------------------------------------------------------
# Explore agreement type codes
# ---------------------------------------------------------------------------

agreement_type_summary = (
    gdf[gdf["AGREENO"].notna()]
    .groupby("AGREETYPE", dropna=False)
    .agg(
        n_rows=("AGREENO", "size"),
        n_agreements=("AGREENO", "nunique"),
        representatives=(
            "DESREP",
            lambda x: sorted(x.dropna().unique().tolist())
        ),
        commencement_dates=(
            "TERMDATE",
            lambda x: sorted(x.dropna().unique().tolist())
        ),
        expiry_dates=(
            "CUREXPTXT",
            lambda x: sorted(x.dropna().unique().tolist())
        ),
        phases=(
            "Phase",
            lambda x: sorted(x.dropna().unique().tolist())
        ),
    )
    .reset_index()
)

display(agreement_type_summary)

,AGREETYPE,n_rows,n_agreements,representatives,commencement_dates,expiry_dates,phases
0,058,27,24,"[ALBERTA POWER (2000) LTD, ALTAGAS LTD., ARC R...","[2022/10/01, 2022/12/01, 2023/01/01]","[2027/10/01, 2027/12/01, 2028/01/01]","[1, 2]"
1,059,6,6,[SHELL CANADA LIMITED],[2011/05/27],[2026/05/27],[2]


In [26]:
# ---------------------------------------------------------------------------
# Inspect temporal fields and missingness
# ---------------------------------------------------------------------------

temporal_fields = [
    "TERMDATE",
    "CONTDATE",
    "CUREXPTXT",
    "VINTAGE",
]

for field in temporal_fields:
    print("\n" + "=" * 70)
    print(field)
    print("=" * 70)

    print(f"Non-null: {gdf[field].notna().sum():,}")
    print(f"Null:     {gdf[field].isna().sum():,}")
    print(f"Unique:   {gdf[field].nunique(dropna=True):,}")

    print("\nValues:")
    print(gdf[field].value_counts(dropna=False).to_string())


TERMDATE
Non-null: 33
Null:     1
Unique:   4

Values:
TERMDATE
2022/12/01    18
2022/10/01     8
2011/05/27     6
2023/01/01     1
None           1

CONTDATE
Non-null: 0
Null:     34
Unique:   0

Values:
CONTDATE
None    34

CUREXPTXT
Non-null: 33
Null:     1
Unique:   4

Values:
CUREXPTXT
2027/12/01    18
2027/10/01     8
2026/05/27     6
2028/01/01     1
None           1

VINTAGE
Non-null: 0
Null:     34
Unique:   0

Values:
VINTAGE
None    34


In [31]:
# ---------------------------------------------------------------------------
# Interactive Leaflet map with Folium
# ---------------------------------------------------------------------------

import folium
import geopandas as gpd

# ---------------------------------------------------------------------------
# Prepare spatial data
# ---------------------------------------------------------------------------

# Keep only records with geometry
spatial_gdf = (
    gdf[gdf.geometry.notna()]
    .copy()
)

# Standardize to the CanCO2Re working CRS
spatial_gdf = spatial_gdf.to_crs("EPSG:3347")

# Calculate map centre in projected CRS
map_center_3347 = spatial_gdf.geometry.union_all().centroid

# Convert centre point to WGS84 for Leaflet
map_center_wgs84 = (
    gpd.GeoSeries(
        [map_center_3347],
        crs="EPSG:3347",
    )
    .to_crs("EPSG:4326")
    .iloc[0]
)

# Leaflet requires EPSG:4326
map_gdf = spatial_gdf.to_crs("EPSG:4326")


# ---------------------------------------------------------------------------
# Build map
# ---------------------------------------------------------------------------

m = folium.Map(
    location=[
        map_center_wgs84.y,
        map_center_wgs84.x,
    ],
    zoom_start=5,
    tiles="OpenStreetMap",
)

folium.GeoJson(
    map_gdf,
    name="Carbon Sequestration Agreements",
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "AGREENO",
            "TRACT",
            "DESREP",
            "ZONE",
            "STATUS",
            "AGREEAREA",
            "TERMDATE",
            "CUREXPTXT",
            "Phase",
        ],
        aliases=[
            "Agreement:",
            "Tract:",
            "Representative:",
            "Zone:",
            "Status:",
            "Current Area (ha):",
            "Commencement:",
            "Current Expiry:",
            "Phase:",
        ],
        sticky=False,
    ),
).add_to(m)

folium.LayerControl().add_to(m)


# ---------------------------------------------------------------------------
# Save HTML
# ---------------------------------------------------------------------------

MAP_OUTPUT = SOURCE_DIR / "AB_carbon_sequestration_agreements.html"

m.save(MAP_OUTPUT)

print(f"Saved interactive map to:\n{MAP_OUTPUT}")

Saved interactive map to:
C:\Users\aviga\Research\potential data\Storage\AB Agreement\AB_carbon_sequestration_agreements.html
